# Optimization of the NACA parameters

This is a test to see how optuna performs to find the best NACA parameters to optimize the aerodynamic finess *(Lift/Drag)*

In [2]:
%load_ext autoreload
%autoreload 2

import sys
import os

sys.path.append(os.path.abspath(".."))

from src.geometry import AirfoilGeometry
from src.physics import AeroSolver
import optuna

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


For this optimization velocity, altitude, chord and alpha are fixed

In [3]:
V_FIXED = 100.0    # m/s
ALT_FIXED = 100     # Niveau de la mer
CHORD_FIXED = 0.5 # 10 cm
ALPHA_TARGET = 5

Only the m, p and t NACA parameters are optimized by Optuna

In [8]:
def objective_naca(trial):
    m = trial.suggest_int("m", 0, 9)
    p = trial.suggest_int("p", 0, 9)
    t = trial.suggest_int("t", 1, 40)
    
    af = AirfoilGeometry(n_points=100)
    xu, yu, xl, yl = af.generate_naca4(m_int=m, p_int=p, t_int=t)
    x_coords, y_coords = af.get_coords_for_solver(xu, yu, xl, yl)
    
    solver = AeroSolver(altitude=ALT_FIXED, velocity=V_FIXED, chord=CHORD_FIXED)
    res = solver.solve(x_coords, y_coords, alpha=ALPHA_TARGET)
    
    cl = res['cl'][0]
    cd = res['cd'][0]
    
    if cd <= 0: return 0
    
    return cl / cd

study = optuna.create_study(direction="maximize")
study.optimize(objective_naca, n_trials=50)

best_naca = f"{study.best_params['m']}{study.best_params['p']}{study.best_params['t']:02d}"
print(f"\nBest : NACA {best_naca}")
print(f"Max Finess (Lift/Drag) : {study.best_value:.2f}")
optuna.visualization.plot_optimization_history(study)

[I 2026-03-09 18:22:29,386] A new study created in memory with name: no-name-a2f2a44e-0bb5-4f05-927e-9f9d4fe7b076
[I 2026-03-09 18:22:29,400] Trial 0 finished with value: 121.41260685727481 and parameters: {'m': 4, 'p': 4, 't': 23}. Best is trial 0 with value: 121.41260685727481.
[I 2026-03-09 18:22:29,408] Trial 1 finished with value: 84.91453913902278 and parameters: {'m': 0, 'p': 8, 't': 12}. Best is trial 0 with value: 121.41260685727481.
[I 2026-03-09 18:22:29,418] Trial 2 finished with value: 60.03499729106925 and parameters: {'m': 6, 'p': 0, 't': 27}. Best is trial 0 with value: 121.41260685727481.
[I 2026-03-09 18:22:29,428] Trial 3 finished with value: 39.21216948691755 and parameters: {'m': 8, 'p': 9, 't': 31}. Best is trial 0 with value: 121.41260685727481.
[I 2026-03-09 18:22:29,441] Trial 4 finished with value: 44.5010639637714 and parameters: {'m': 5, 'p': 0, 't': 12}. Best is trial 0 with value: 121.41260685727481.
[I 2026-03-09 18:22:29,449] Trial 5 finished with value:


Best : NACA 9510
Max Finess (Lift/Drag) : 215.77


In [5]:
optuna.visualization.plot_contour(study, params=['m', 'p', 't'])


In [6]:
optuna.visualization.plot_param_importances(study)